# MSAI635: Automating GPU Optimizations using RL
## End-to-End Multi-Turn GRPO Training and Evaluation
This notebook runs the full pipeline: environment checks, teacher verification, prompt inspection, optional SFT handoff, multi-turn policy training, and evaluation on training and holdout tasks. It targets Google Colab and Zaratan style GPU nodes.

### 1. Hardware and Environment Verification
Confirm that a CUDA GPU is visible and configure the PyTorch allocator to reduce fragmentation.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi

### 2. Dependency Installation
Install the pinned training stack and the local package. Restart the runtime after installation if Colab asks for it.

In [ ]:
!pip install -q transformers==4.57.6 peft==0.17.1 bitsandbytes==0.48.1 \
    trl==0.23.1 ninja pybind11 accelerate datasets scipy
!pip install -q -e .

### 3. Toolchain and Inline Compilation Sanity Check
Compile a small CUDA extension with the same code path used by the verifier.

In [ ]:
import torch

from engine.compiler import compile_kernel_ex

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active device: {torch.cuda.get_device_name(0)}")

sanity_code = """
#include "torch/extension.h"
#include "cuda_runtime.h"

__global__ void add_kernel(const float* x, float* out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) out[idx] = x[idx] + 1.0f;
}

torch::Tensor run_cuda(torch::Tensor x) {
    auto out = torch::empty_like(x);
    int n = x.numel();
    const float* in_ptr = (const float*)x.data_ptr();
    float* out_ptr = (float*)out.data_ptr();
    add_kernel<<<(n + 255) / 256, 256>>>(in_ptr, out_ptr, n);
    return out;
}
"""

result = compile_kernel_ex(sanity_code)
print(f"Compile status: {result.status}")
print(result.message)

### 4. Configuration
Load the multi-turn YAML config and point outputs at a Colab friendly location. On a single GPU Colab runtime, set the verifier device to the same device as training.

In [ ]:
from configs.grpo_config import ExperimentConfig

config = ExperimentConfig.from_yaml("configs/multiturn_grpo.yaml")
config.use_dev_model = True
config.use_tiny_model = False
config.device = "cuda:0"
config.verifier_device = config.device
config.use_verifier_worker = False
config.output_dir_arm_a = "./outputs/colab_arm_a"
config.output_dir_arm_b = "./outputs/colab_arm_b"
config.eval_output_dir = "./outputs/colab_eval"
config.archive_path = "./outputs/colab_archive.jsonl"
config.telemetry_path = "./logs/colab_telemetry.jsonl"
config.resolve_paths()

print("mode:", config.mode)
print("training tasks:", config.training_tasks)
print("holdout tasks:", config.holdout_tasks)
print("training device:", config.device)
print("verifier device:", config.verifier_device)

### 5. Teacher Kernel Verification
Compile, verify, and benchmark the stored teacher examples. Holdout tasks now include teachers, so Arm A receives contrastive context on both training and holdout tasks.

In [ ]:
from data.example_store import get_teacher_examples
from engine.benchmark import measure_speedup
from engine.compiler import compile_kernel_ex
from engine.verifier import verify_kernel

for task_id in config.eval_tasks:
    for teacher in get_teacher_examples(task_id):
        compiled = compile_kernel_ex(teacher["code"])
        if compiled.module is None:
            print(f"[FAIL] {task_id}/{teacher['name']}: {compiled.message}")
            continue
        verification = verify_kernel(compiled.module, task_id, device=config.verifier_device)
        if not verification.correct:
            print(f"[FAIL] {task_id}/{teacher['name']}: {verification.message}")
            continue
        speedup, stable = measure_speedup(
            compiled.module,
            task_id,
            device=config.verifier_device,
        )
        print(f"[PASS] {task_id}/{teacher['name']}: {speedup:.2f}x stable={stable}")

### 6. Inspect Prompt Construction
Compare the reference-only prompt with the contrastive prompt and inspect a second-turn refinement message.

In [ ]:
from engine.prompt_builder import build_prompt, build_turn_messages

print("Reference only prompt sample")
print(build_prompt("softmax", mode="reference_only")[:400])
print("Contrastive prompt sample")
print(build_prompt("softmax", mode="contrastive")[:600])

messages = build_turn_messages(
    "softmax",
    mode="reference_only",
    turn_index=1,
    best_code="__global__ void kernel() {}",
    feedback="Compilation failed: missing include.",
    speedup=None,
)
print("Second turn message")
print(messages[-1]["content"][:600])

### 7. Optional SFT Handoff
Set `sft_adapter_path` to a teammate supplied SFT adapter directory or to a merged full model checkpoint. Leave it empty to train directly from the base model.

In [ ]:
from pathlib import Path

sft_adapter_path = ""

if sft_adapter_path and Path(sft_adapter_path).exists():
    config.sft_adapter_path = sft_adapter_path
    print("Using SFT handoff:", sft_adapter_path)
else:
    config.sft_adapter_path = None
    print("No SFT handoff supplied. Training starts from the base model.")

### 8. Dry Run
Exercise one rollout and one optimizer step end to end before starting a long run.

In [ ]:
import gc

import torch

from train_grpo import run_training

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

config.max_steps = 1
config.num_turns = 1
config.save_steps = 0

dry_run_adapter = run_training(
    "arm_a",
    config,
    dry_run=True,
    dispatcher_kind="in_process",
)
print("Dry run adapter:", dry_run_adapter)

### 9. Multi-Turn Training
Train both matched arms. Arm A uses contrastive prompts and Arm B uses reference-only prompts. Pass `resume_from_checkpoint` after a preemption to continue from the latest checkpoint.

In [ ]:
import gc

import torch

config.mode = "multi_turn"
config.max_steps = 20
config.num_turns = 4
config.save_steps = 10
resume_from_checkpoint = ""

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

arm_a_path = run_training(
    "arm_a",
    config,
    dispatcher_kind="auto",
    resume_from_checkpoint=resume_from_checkpoint or None,
)
print("Arm A adapter:", arm_a_path)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

arm_b_path = run_training(
    "arm_b",
    config,
    dispatcher_kind="auto",
    resume_from_checkpoint=resume_from_checkpoint or None,
)
print("Arm B adapter:", arm_b_path)

### 10. Evaluation on Training and Holdout Tasks
Evaluate both adapters with the configured multi-turn budget. The base model control can be run by passing `adapter_path=None`.

In [ ]:
import gc

import torch

from evaluate import evaluate_checkpoint
from models.loader import load_model_and_tokenizer
from src.pipeline.env_dispatcher import build_dispatcher

dispatcher = build_dispatcher(config, kind="auto")
results = {}

for arm, adapter_path, prompt_mode in (
    ("arm_a", arm_a_path, "contrastive"),
    ("arm_b", arm_b_path, "reference_only"),
):
    model, tokenizer = load_model_and_tokenizer(
        config=config,
        is_trainable=False,
        adapter_path=adapter_path,
    )
    results[arm] = evaluate_checkpoint(
        model=model,
        tokenizer=tokenizer,
        task_keys=config.eval_tasks,
        prompt_mode=prompt_mode,
        config=config,
        dispatcher=dispatcher,
        num_samples=1,
    )
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

dispatcher.close()

### 11. Statistical Comparison
Summarize correctness, speedup thresholds, and the holdout split for both arms.

In [ ]:
import json

import pandas as pd

rows = []
for arm in ("arm_a", "arm_b"):
    summary = results[arm]
    holdout = summary.get("by_split", {}).get("holdout", {})
    rows.append(
        {
            "Arm": arm,
            "Correctness": summary["correctness_rate"],
            "Fast 1.0x": summary["fast_1_0_rate"],
            "Fast 1.1x": summary["fast_1_1_rate"],
            "Median speedup": summary["median_speedup"],
            "Holdout correctness": holdout.get("correctness_rate", 0.0),
            "Holdout median speedup": holdout.get("median_speedup", 0.0),
        }
    )

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

with open("outputs/colab_eval/comparison.json", "w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2)